<a href="https://colab.research.google.com/github/zankuit/NT120-labs/blob/main/NT120_Session03_Lab_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NT120 Lab 3: Python, Google Colab and Tabular Data

Introduction to AI for Computer Networks and Cybersecurity, Faculty of Computer Networks and Communications, UIT (VNU-HCM). Course page: https://link.uit.edu.vn/NT120

This notebook is the guided practice for Session 3. By the end of it you will be able to:

- run and modify a notebook in Google Colab, and explain the notebook contract,
- use the Python constructs that data work needs,
- inspect a tabular dataset with Pandas: shape, types, distributions and missing values,
- select, filter, group and join tabular data,
- apply the five reproducibility rules used throughout this course.

### How to use this notebook

If you opened it from a shared link, use File > Save a copy in Drive first, so that your edits are kept. Run a cell with Shift+Enter.

Cells titled "Your turn" contain a TODO for you to complete, and every other cell is ready to run. The self-check cells print PASS or TODO and never stop the notebook, so it always runs from top to bottom even when it is unfinished. That property is deliberate: it lets you use Runtime > Restart session and run all as your final test before you submit.

| Minutes | Section |
|---|---|
| 10 to 20 | 1. Setup and the notebook contract |
| 20 to 55 | 2. Python essentials, 3. NumPy, 4. Pandas basics, 5. Selecting, filtering, grouping and joining |
| 55 to 60 | 6. Checkpoint |
| 60 to 82 | 7. In-class activity: guided data inspection |
| 82 to 90 | 8. Reproducibility rules and wrap-up |
| after class | Homework and optional stretch |

## 1. Setup and the notebook contract

Colab is a Jupyter notebook running on a temporary virtual machine that Google lends you. Python, NumPy, Pandas, Matplotlib and Scikit-learn are already installed, so there is nothing to set up. The price is that the machine is discarded after a period of inactivity, and everything you wrote to its disk goes with it.

The first code cell of every notebook in this course looks like the one below: imports, a fixed random seed, and the library versions printed for the record.

In [1]:
# Cell 1 of every notebook in this course: imports, seed, versions.
import os
import sys
import time

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

print("python    ", sys.version.split()[0])
print("numpy     ", np.__version__)
print("pandas    ", pd.__version__)
print("matplotlib", matplotlib.__version__)
print("seed      ", RANDOM_STATE)

python     3.13.15
numpy      2.1.3
pandas     2.2.3
matplotlib 3.10.0
seed       42


In [2]:
#@title Course helpers: run once, do not edit { display-mode: "form" }
import hashlib

_RESULTS = {}
_KEYS = {'Q1': '28acc51596', 'Q2': 'fa500411cf', 'Q3': 'd29cc3a88b', 'Q4': 'c1e2939ab5', 'Q5': '1f6a883eb9', 'Q6': 'd6748e217e', 'Q6 fix': '862aecd520'}


def _canon(qid, value):
    """Turn an answer into a canonical string, so equivalent answers hash alike."""
    if qid == "Q1":
        rows, cols = value
        return f"{int(rows)},{int(cols)}"
    if qid == "Q2":
        return "|".join(f"{k}:{float(v):.4f}" for k, v in sorted(dict(value).items()))
    if qid == "Q3":
        if isinstance(value, pd.Series):
            value = value.index
        return ",".join(str(int(p)) for p in list(value))
    if qid == "Q4":
        return "|".join(f"{k}:{float(v):.2f}" for k, v in sorted(dict(value).items()))
    if qid == "Q5":
        if isinstance(value, pd.Series):
            value = value.sort_values(ascending=False)
            name, count = value.index[0], value.iloc[0]
        else:
            name, count = value
        return f"{name}:{int(count)}"
    if qid == "Q6":
        column, target = value
        kind = str(target).lower()
        kind = "numeric" if any(t in kind for t in ("int", "float", "num")) else kind
        return f"{column}:{kind}"
    if qid == "Q6 fix":
        if not pd.api.types.is_numeric_dtype(value):
            return "not numeric"
        return f"numeric:{int(round(float(value.sum())))}"
    raise KeyError(qid)


def check(qid, value):
    """Compare an answer with the stored fingerprint. It tells you if you are right, not what is right."""
    if value is None:
        _RESULTS[qid] = False
        print(f"{qid}: not attempted yet (the answer is still None).")
        return
    try:
        fingerprint = hashlib.sha256(_canon(qid, value).encode()).hexdigest()[:10]
    except Exception as exc:
        _RESULTS[qid] = False
        print(f"{qid}: could not read your answer ({type(exc).__name__}). Check its type and shape.")
        return
    _RESULTS[qid] = fingerprint == _KEYS[qid]
    print(f"{qid}: " + ("correct." if _RESULTS[qid] else "not quite. Re-read the question, then check units and rounding."))


def test(name, condition):
    """Record a PASS or TODO for a small exercise. `condition` may be a value or a function."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception:
        ok = False
    _RESULTS[name] = ok
    print(("PASS  " if ok else "TODO  ") + name)


def report():
    """Summarize every self-check that has run so far."""
    passed = [k for k, ok in _RESULTS.items() if ok]
    todo = [k for k, ok in _RESULTS.items() if not ok]
    print(f"Self-checks passed: {len(passed)} of {len(_RESULTS)}")
    if todo:
        print("Still open:", ", ".join(todo))

In [3]:
IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)
print("Working directory:", os.getcwd())
print("Files here:", sorted(os.listdir(".")))

Running in Colab: True
Working directory: /content
Files here: ['.config', 'sample_data']


Files that must survive the session belong on Google Drive, not on the runtime. Mounting Drive takes two lines. They are commented out here so that the notebook also runs outside Colab; remove the `#` signs in Colab, approve the prompt, and save to a path under `/content/drive/MyDrive/`.

In [4]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


### Hidden state

Run the next two cells in order. The second one works only because the first one ran before it.

In [5]:
threshold = 5

In [ ]:
scores = [3, 7, 9, 2, 6]
print("alerts:", sum(s > threshold for s in scores))

alerts: 3


Now try to break it. Use Runtime > Restart session, then run only the second cell and read the error. The variable `threshold` lived in memory, and the restart erased it. In a real notebook the cell that created a variable is often edited or deleted long after the variable was made, so the notebook keeps working for you and fails for everyone else. The rule that prevents it is the notebook contract: the notebook must produce every result when run from the top on a fresh runtime. Run all the cells again from the top (Runtime > Run all) before you continue.

## 2. Python essentials for data work

You need a small subset of Python: lists, dictionaries, comprehensions and functions. The cell below shows all four.

In [6]:
ports = [22, 80, 443, 3389]
names = {22: "ssh", 80: "http", 443: "https"}

known = [p for p in ports if p in names]           # keep the ports we can name
labels = [names.get(p, "other") for p in ports]    # name every port, "other" if unknown
print(known)
print(labels)


def rate(bytes_, seconds):
    """Bytes per second, defined as 0.0 for a flow that lasted no time."""
    if seconds <= 0:
        return 0.0
    return bytes_ / seconds


print(rate(1500, 0.5), rate(1500, 0))

[22, 80, 443]
['ssh', 'http', 'https', 'other']
3000.0 0.0


### Your turn 2.1: comprehension, dictionary, function

Complete the three items. Each self-check prints PASS when your result matches.

In [11]:

# A1. Keep only the ports that carry unencrypted traffic: 21, 23, 80 and 8080.
observed_ports = [22, 80, 443, 21, 8080, 3389, 23, 80]
unencrypted = [p for p in observed_ports if p in [21, 23, 80, 8080]]      # TODO: one list comprehension over observed_ports
test("A1 unencrypted ports", unencrypted == [80, 21, 8080, 23, 80])

# A2. Count flows per protocol with a dictionary and a loop.
protocols = ["TCP", "UDP", "TCP", "ICMP", "TCP", "UDP"]
counts = {}
for p in protocols:
    counts[p] = counts.get(p, 0) + 1                # TODO: add one to counts[p]; dict.get(key, 0) helps
test("A2 protocol counts", counts == {"TCP": 3, "UDP": 2, "ICMP": 1})


# A3. Throughput in megabits per second; 0.0 when seconds <= 0.
def mbps(bytes_, seconds):
    if seconds <= 0:
        return 0.0
    return bytes_ * 8 / 1e6 / seconds        # TODO: bytes * 8 / 1e6 / seconds, guarded like rate()


test("A3 mbps", lambda: mbps(1_000_000, 8) == 1.0 and mbps(500, 0) == 0.0)


PASS  A1 unencrypted ports
PASS  A2 protocol counts
PASS  A3 mbps


## 3. NumPy: arrays, shapes and vectorized thinking

A Python list can hold anything and is slow. A NumPy array holds one type in a fixed shape, and that restriction is what makes it fast. Read the three properties `shape`, `dtype` and `axis` off the array below.

In [12]:
a = np.array([[4.12,   9,   1842, 0.32],
              [0.03,   2,      0, 0.00],
              [61.5, 318, 402110, 0.71]])

print("shape:", a.shape, "| dtype:", a.dtype, "| ndim:", a.ndim, "| size:", a.size)
print("axis 0 sums (one per column):", a.sum(axis=0))
print("axis 1 sums (one per row)   :", a.sum(axis=1))

shape: (3, 4) | dtype: float64 | ndim: 2 | size: 12
axis 0 sums (one per column): [6.56500e+01 3.29000e+02 4.03952e+05 1.03000e+00]
axis 1 sums (one per row)   : [1.8554400e+03 2.0300000e+00 4.0249021e+05]


`sum(axis=0)` collapses axis 0, the rows, and leaves one number per column. Keep that reading in mind: the axis you name is the one that disappears.

### Loops versus vectorized expressions

The habit from C is to loop over the index. NumPy lets you state the definition of the quantity instead. The cell below computes bytes per second for a million simulated flows both ways.

In [13]:
gen = np.random.default_rng(RANDOM_STATE)
byte_counts = gen.integers(0, 10_000, 1_000_000).astype(float)
durations = gen.exponential(2.0, 1_000_000)
durations[gen.random(1_000_000) < 0.05] = 0.0        # 5% of the flows lasted no time

t0 = time.perf_counter()
rates_loop = []
for i in range(len(byte_counts)):
    if durations[i] > 0:
        rates_loop.append(byte_counts[i] / durations[i])
    else:
        rates_loop.append(0.0)
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
with np.errstate(divide="ignore", invalid="ignore"):
    rates_vec = np.where(durations > 0, byte_counts / durations, 0.0)
t_vec = time.perf_counter() - t0

print(f"loop: {t_loop:.3f} s | vectorized: {t_vec:.4f} s | about {t_loop / t_vec:.0f} times faster")
print("same answer:", np.allclose(rates_loop, rates_vec))

loop: 0.646 s | vectorized: 0.0107 s | about 60 times faster
same answer: True


`np.where` evaluates both branches before it chooses, so the division by zero still happens and NumPy would print a warning. The `np.errstate` block silences it because we handle those entries on purpose. Remove the block once and read the warning, so you recognize it later.

### Your turn 3.1: broadcasting and guarded division

In [14]:

# B1. Divide every entry of `a` by its column total, so that each column sums to 1.
share = a / a.sum(axis=0)            # TODO: one expression using a and a.sum(axis=0)
test("B1 column shares", lambda: np.allclose(share.sum(axis=0), 1.0))

# B2. Standardize each column of `a`: subtract its mean, divide by its standard deviation.
z = (a - a.mean(axis=0)) / a.std(axis=0)               # TODO: (a - mean) / std, each computed along axis=0
test("B2 z-scores", lambda: np.allclose(z.mean(axis=0), 0) and np.allclose(z.std(axis=0), 1))

# B3. Bytes per packet, 0 where there are no packets, with no loop and no warning.
pkts = np.array([10, 0, 4, 0, 25])
byts = np.array([1500, 0, 320, 0, 9000])
bytes_per_pkt = np.where(pkts > 0, byts / np.maximum(pkts, 1), 0.0)    # TODO: np.where, and a denominator that is never zero
test("B3 bytes per packet", lambda: np.allclose(bytes_per_pkt, [150, 0, 80, 0, 360]))


PASS  B1 column shares
PASS  B2 z-scores
PASS  B3 bytes per packet


## 4. Pandas: loading and inspecting a table

Pandas gives NumPy arrays labeled rows and named columns. A `DataFrame` is the whole table, and a single column is a `Series` with one dtype.

We practice the tools on a tiny table where you can verify every number by eye. In Section 7 you point the same tools at the full course table.

The cell below writes a small CSV file, as if a colleague had sent it to you, and reads it back with `read_csv`.

In [15]:
flows_small = {
    "dst_port":  [443, 22, 80, 443, 53, 22, 8080, 443],
    "protocol":  ["TCP", "TCP", "TCP", "TCP", "UDP", "TCP", "TCP", "TCP"],
    "duration":  [4.12, 0.03, 61.5, 12.8, 0.0, 0.02, 33.4, 7.9],
    "fwd_pkts":  [9, 2, 318, 41, 1, 2, 120, 25],
    "fwd_bytes": [1842, 120, 402110, 15300, 96, 120, 88750, 6120],
    "ttl":       ["64", "64", "128", "64", "unknown", "64", "128", "64"],
    "label":     ["BENIGN", "PortScan", "BENIGN", "BENIGN", "BENIGN", "PortScan", "BENIGN", "BENIGN"],
}
tmp = pd.DataFrame(flows_small)
tmp["flow_bytes_s"] = tmp["fwd_bytes"] / tmp["duration"].replace(0, np.nan)   # no rate for a 0 s flow
tmp.to_csv("flows_small.csv", index=False)

small = pd.read_csv("flows_small.csv")
small

,dst_port,protocol,duration,fwd_pkts,fwd_bytes,ttl,label,flow_bytes_s
0,443,TCP,4.12,9,1842,64,BENIGN,447.087379
1,22,TCP,0.03,2,120,64,PortScan,4000.000000
2,80,TCP,61.50,318,402110,128,BENIGN,6538.373984
3,443,TCP,12.80,41,15300,64,BENIGN,1195.312500
4,53,UDP,0.00,1,96,unknown,BENIGN,NaN
5,22,TCP,0.02,2,120,64,PortScan,6000.000000
6,8080,TCP,33.40,120,88750,128,BENIGN,2657.185629
7,443,TCP,7.90,25,6120,64,BENIGN,774.683544


### Six lines that characterize any new CSV

Run them one at a time and read each output before you move on.

In [16]:
small.shape          # (rows, columns): how much data is there?

(8, 8)

In [17]:
small.info()         # dtypes and non-null counts: what is mistyped or missing?

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   dst_port      8 non-null      int64  
 1   protocol      8 non-null      object 
 2   duration      8 non-null      float64
 3   fwd_pkts      8 non-null      int64  
 4   fwd_bytes     8 non-null      int64  
 5   ttl           8 non-null      object 
 6   label         8 non-null      object 
 7   flow_bytes_s  7 non-null      float64
dtypes: float64(2), int64(3), object(3)
memory usage: 644.0+ bytes


In [18]:
small.describe()     # min, max, mean and quartiles: ranges and skew

,dst_port,duration,fwd_pkts,fwd_bytes,flow_bytes_s
count,8.000000,8.000000,8.000000,8.000000,7.000000
mean,1198.250000,14.971250,64.750000,64307.250000,3087.520434
std,2787.700012,21.869513,109.791686,139775.642043,2493.668199
min,22.000000,0.000000,1.000000,96.000000,447.087379
25%,45.250000,0.027500,2.000000,120.000000,984.998022
50%,261.500000,6.010000,17.000000,3981.000000,2657.185629
75%,443.000000,17.950000,60.750000,33662.500000,5000.000000
max,8080.000000,61.500000,318.000000,402110.000000,6538.373984


In [20]:
small["label"].value_counts(normalize=True)     # class balance, before anything else

,proportion
label,
BENIGN,0.75
PortScan,0.25


In [21]:
small.isna().sum().sort_values(ascending=False)     # missing values, worst column first

,0
flow_bytes_s,1
dst_port,0
duration,0
protocol,0
fwd_pkts,0
fwd_bytes,0
ttl,0
label,0


In [22]:
small.head(3)        # sanity check: does a row look like a row?

,dst_port,protocol,duration,fwd_pkts,fwd_bytes,ttl,label,flow_bytes_s
0,443,TCP,4.12,9,1842,64,BENIGN,447.087379
1,22,TCP,0.03,2,120,64,PortScan,4000.000000
2,80,TCP,61.50,318,402110,128,BENIGN,6538.373984


Two things in those outputs deserve a second look. `info()` reports `ttl` as text even though it looks like a number (Pandas 3 calls that dtype `str`, and earlier versions call it `object`), and `describe()` silently skipped it for the same reason. A column that is missing from `describe()` is a column whose type you should question. Here the culprit is the string `"unknown"`: a single non-numeric entry makes Pandas read the whole column as text.

Every change to the data happens in code, so the fix goes in a cell and not in a spreadsheet editor. `pd.to_numeric(..., errors="coerce")` turns anything it cannot parse into a missing value.

In [23]:
small["ttl"] = pd.to_numeric(small["ttl"], errors="coerce")
small.dtypes

,0
dst_port,int64
protocol,object
duration,float64
fwd_pkts,int64
fwd_bytes,int64
ttl,float64
label,object
flow_bytes_s,float64


## 5. Selecting, filtering, grouping and joining

### Selection and filtering

In [24]:
dur = small["duration"]                              # one column is a Series
sub = small[["duration", "fwd_pkts", "label"]]       # a list of columns is a DataFrame
print(type(dur).__name__, type(sub).__name__)

# boolean masks: read them out loud
attacks = small[small["label"] != "BENIGN"]
web = small[small["dst_port"].isin([80, 443, 8080])]
odd = small[(small["protocol"] == "UDP") | (small["duration"] < 0.05)]
print(len(attacks), len(web), len(odd))

# label-based and position-based access
small.loc[small["label"] == "PortScan", ["duration", "fwd_pkts"]]

Series DataFrame
2 5 3


,duration,fwd_pkts
1,0.03,2
5,0.02,2


In [26]:
small.iloc[0:3, 0:4]        # by position: first three rows, first four columns

,dst_port,protocol,duration,fwd_pkts
0,443,TCP,4.12,9
1,22,TCP,0.03,2
2,80,TCP,61.50,318


Combine masks with `&` and `|`, and put each condition in parentheses. Python's `and` does not work element by element, and Pandas tells you so.

In [27]:
try:
    small[(small["duration"] > 10) and (small["protocol"] == "TCP")]
except ValueError as err:
    print("ValueError:", err)

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().


Prefer `.loc` with names over `.iloc` with positions. Column order changes when a file is re-exported, and names do not.

### Group, apply, combine

In [25]:
# how does each feature differ between the classes?
small.groupby("label")[["duration", "fwd_bytes"]].agg(["mean", "median"])

duration         fwd_bytes         
               mean  median      mean   median
label                                         
BENIGN    19.953333  10.350   85703.0  10710.0
PortScan   0.025000   0.025     120.0    120.0

In [ ]:
# the busiest destination ports, and the share of attack flows on each port
small["dst_port"].value_counts().head(3)

In [ ]:
small.groupby("dst_port")["label"].apply(lambda s: (s != "BENIGN").mean())

In [ ]:
mean_duration = small.groupby("label")["duration"].mean()

ax = mean_duration.plot.bar(rot=0, color=["tab:green", "tab:red"], logy=True)
ax.set_ylabel("mean duration (seconds, log scale)")
ax.set_title("Mean flow duration per class")
plt.show()

Figure 1. Mean flow duration per class in the eight-flow practice table, on a logarithmic axis so that both bars are visible. The values are illustrative and only demonstrate the plotting calls. Rule 4 of the course: every figure says what it shows, in a caption or a title.

### Joining an external table

A `merge` attaches context that is not in the flow record. Here we add a service name for each port.

In [ ]:
services = pd.DataFrame({"dst_port": [22, 80, 443], "service": ["ssh", "http", "https"]})

joined = small.merge(services, on="dst_port", how="left")
joined["service"] = joined["service"].fillna("other")
print("rows before:", len(small), "| rows after:", len(joined))
joined[["dst_port", "service", "label"]]

`how="left"` keeps every flow even when the port is unknown. Always compare the row count before and after a merge, because a duplicate key in the second table silently multiplies rows.

In [ ]:
services_bad = pd.DataFrame({"dst_port": [22, 80, 443, 443],
                             "service": ["ssh", "http", "https", "https-alt"]})

bad = small.merge(services_bad, on="dst_port", how="left")
print("rows before:", len(small), "| rows after:", len(bad), "  <- three flows appeared from nowhere")

try:
    small.merge(services_bad, on="dst_port", how="left", validate="many_to_one")
except pd.errors.MergeError as err:
    print("MergeError:", err)

`validate="many_to_one"` turns that silent mistake into an error. Use it whenever the right-hand table is meant to be a lookup.

### Your turn 5.1: filter, group, join

In [ ]:

# D1. The flows that are TCP and last longer than 10 seconds.
long_tcp = None         # TODO: a boolean mask with two conditions joined by &
test("D1 long TCP flows", lambda: len(long_tcp) == 3 and set(long_tcp["dst_port"]) == {80, 443, 8080})

# D2. Mean and median of duration and fwd_bytes for each label, in one table.
by_label = None         # TODO: groupby, select two columns, then .agg([...])
test("D2 group table", lambda: by_label.shape == (2, 4))

# D3. Attach `services` to every flow, label unknown ports "other", and keep all 8 rows.
with_service = None     # TODO: merge with how="left", then fillna on the service column
test("D3 join keeps rows", lambda: len(with_service) == len(small) and with_service["service"].notna().all())


## 6. Checkpoint: predict the output

Write your prediction for each of the three snippets in the cell after them **before** you run anything. Then run the snippets and compare.

In [ ]:
snippet = pd.DataFrame({"port": [22, 80, 22, 443], "bytes": [100, 250, 90, 500]})

# 1
print(snippet.shape)

# 2
print(snippet[snippet["port"] == 22]["bytes"].mean())

# 3
print(snippet.groupby("port")["bytes"].sum().loc[22])


My predictions.

1.
2.
3.


Answer the three questions in words. Double-click a cell to edit it.

1. What does `df.info()` tell you that `df.describe()` does not?
2. Why does a notebook that only works when its cells are run out of order fail the reproducibility rule?
3. Which Pandas call counts the records per label?


Your answers.

1.
2.
3.


## 7. In-class activity: guided data inspection

You have twenty minutes. Working alone or in pairs, answer the six questions below with code, not by scrolling. Each question ends with a self-check that tells you whether your answer is right without revealing it. Compare your code with your neighbor's afterward, because there are usually three reasonable ways to write each one.

### The course table

The next cell builds the running dataset of Sessions 3 to 10 and writes it to `flows.csv` on the Colab disk, so that everyone starts from the same file. Each row is one bidirectional network flow, and the `label` column says whether the flow is `BENIGN` or a `PortScan`. Session 4 explains where such rows come from. For now, the goal is to be able to look at them.

The values are simulated for teaching. The table has the same structure as the CIC-IDS style CSV that the course uses (measurements per flow, a `protocol` column and a `label`), but it should not be used to draw conclusions about real attack traffic. If your instructor gives you a real CSV, upload it to the Colab file browser, set `DATA_PATH` to its name, and the rest of the notebook runs unchanged. You do not need to read the generator.

In [ ]:
#@title Data generator: run, do not edit { display-mode: "form" }
def _fit_log_duration(z, target, sd=0.9, lo=-3.0, hi=np.log10(120.0)):
    """Durations with 10**(mu + sd*z), capped to [1 ms, 120 s]; mu is tuned so the mean equals `target`."""
    a, b = -3.0, 3.0
    for _ in range(60):
        mu = (a + b) / 2
        if (10 ** np.clip(mu + sd * z, lo, hi)).mean() < target:
            a = mu
        else:
            b = mu
    return 10 ** np.clip(mu + sd * z, lo, hi)


def make_flow_table(n_rows=112_450, n_attack=416, n_zero=152, seed=42):
    """Simulate a teaching-sized table of bidirectional flow records (CIC-IDS style)."""
    rng = np.random.default_rng(seed)
    n = n_rows
    n_ben = n - n_attack

    # ---- class, protocol and destination port ------------------------------
    label = np.array(["BENIGN"] * n_ben + ["PortScan"] * n_attack, dtype=object)
    is_scan = label == "PortScan"

    proto = np.empty(n, dtype=object)
    proto[:n_ben] = rng.choice(["TCP", "UDP", "ICMP"], n_ben, p=[0.78, 0.21, 0.01])
    proto[n_ben:] = "TCP"
    tcp = proto == "TCP"

    port = np.zeros(n, dtype=np.int64)
    ephemeral = rng.integers(1024, 65536, n)
    tcp_ports = np.array([443, 80, 8080, 22, 3389, 445, 25, 993, 21])
    tcp_p = [0.38, 0.27, 0.10, 0.07, 0.05, 0.05, 0.03, 0.03, 0.02]
    udp_ports = np.array([53, 123, 443, 5353, 1900])
    udp_p = [0.62, 0.16, 0.12, 0.06, 0.04]
    for name, ports, p in (("TCP", tcp_ports, tcp_p), ("UDP", udp_ports, udp_p)):
        m = (proto == name) & ~is_scan
        pick = rng.choice(ports, m.sum(), p=p)
        port[m] = np.where(rng.random(m.sum()) < 0.12, ephemeral[m], pick)
    port[is_scan] = rng.integers(1, 1025, n_attack)      # a scan sweeps low ports

    # ---- duration in seconds -----------------------------------------------
    dur = np.zeros(n)
    zero_b = np.zeros(n_ben, dtype=bool)
    zero_b[rng.choice(n_ben, n_zero, replace=False)] = True   # single-packet flows
    z = rng.standard_normal(n_ben)
    long_b = _fit_log_duration(z[~zero_b], target=18.40 * n_ben / (n_ben - n_zero))
    dur_b = np.zeros(n_ben)
    dur_b[~zero_b] = long_b
    dur[:n_ben] = dur_b                                       # overall benign mean is 18.40 s
    g = rng.gamma(4.0, 1.0, n_attack)
    dur[n_ben:] = g * (0.04 / g.mean())                       # scans are very short
    zero = dur == 0
    assert zero.sum() == n_zero

    # ---- packet and byte counts --------------------------------------------
    fp = np.ones(n, dtype=np.int64)
    bp = np.zeros(n, dtype=np.int64)
    ben = ~is_scan & ~zero
    fp[ben] = 1 + rng.poisson(2.0 + 2.5 * np.sqrt(dur[ben]))
    bp[ben] = rng.poisson(fp[ben] * rng.uniform(0.4, 1.6, ben.sum()))
    fp[is_scan] = rng.choice([1, 2, 3], n_attack, p=[0.7, 0.2, 0.1])
    bp[is_scan] = rng.choice([0, 1], n_attack, p=[0.5, 0.5])

    fl = np.zeros(n)                                      # mean payload per fwd packet
    bl = np.zeros(n)                                      # mean payload per bwd packet
    fl[ben] = np.minimum(rng.lognormal(4.8, 0.9, ben.sum()), 1460)
    bl[ben] = np.minimum(rng.lognormal(5.6, 1.0, ben.sum()), 1460)
    fl[zero] = np.minimum(rng.lognormal(4.5, 0.6, zero.sum()), 1460)
    fwd_bytes = np.round(fp * fl).astype(np.int64)
    bwd_bytes = np.round(bp * bl).astype(np.int64)
    tot_pkts = fp + bp
    tot_bytes = fwd_bytes + bwd_bytes

    # ---- rates -------------------------------------------------------------
    flow_bytes_s = tot_bytes / np.where(zero, 1.0, dur)
    flow_bytes_s[zero] = np.nan       # the export step turned 0-second divisions into gaps
    safe = np.maximum(dur, 1e-3)      # ...but this rate was guarded, so it has no gaps

    # ---- packet length statistics ------------------------------------------
    def length_stats(m):
        mx = np.minimum(m * rng.uniform(1.0, 3.0, n), 1460)
        mn = m * rng.uniform(0.0, 1.0, n)
        sd = m * rng.uniform(0.0, 0.8, n)
        return mx, mn, sd

    f_mx, f_mn, f_sd = length_stats(fl)
    b_mx, b_mn, b_sd = length_stats(bl)
    p_mean = tot_bytes / tot_pkts
    p_sd = np.maximum(f_sd, b_sd) * rng.uniform(0.8, 1.2, n)

    # ---- inter-arrival times -----------------------------------------------
    def iat_stats(mean, k):
        sd = np.where(k > 2, mean * rng.uniform(0.5, 2.0, n), 0.0)
        mx = np.where(k > 1, mean * rng.uniform(1.5, 6.0, n), 0.0)
        mn = np.where(k > 1, mean * rng.uniform(0.0, 0.5, n), 0.0)
        return sd, mx, mn

    fl_iat = dur / np.maximum(tot_pkts - 1, 1)
    f_iat = np.where(fp > 1, dur / np.maximum(fp - 1, 1), 0.0)
    b_iat = np.where(bp > 1, dur / np.maximum(bp - 1, 1), 0.0)
    fl_sd, fl_mx, fl_mn = iat_stats(fl_iat, tot_pkts)
    f_isd, f_imx, f_imn = iat_stats(f_iat, fp)
    b_isd, b_imx, b_imn = iat_stats(b_iat, bp)

    # ---- TCP flags and headers ---------------------------------------------
    real = tcp & ~is_scan
    syn = np.where(tcp, np.where(is_scan, 1, rng.integers(1, 3, n)), 0)
    fin = np.where(real & (tot_pkts > 3), rng.integers(0, 3, n), 0)
    rst = np.where(is_scan & (bp > 0), 1, np.where(real & (rng.random(n) < 0.03), 1, 0))
    psh = np.where(real, rng.poisson(fp * 0.3), 0)
    ack = np.where(real, np.maximum(tot_pkts - 1, 0), 0)
    urg = np.where(real & (rng.random(n) < 0.01), 1, 0)
    ece = np.where(tcp & (rng.random(n) < 0.005), 1, 0)
    hdr = np.where(tcp, 32, 8)

    # ---- bulk transfer, windows, active and idle periods -------------------
    f_bulk = (fp >= 6) & (rng.random(n) < 0.4) & ~is_scan
    b_bulk = (bp >= 6) & (rng.random(n) < 0.4) & ~is_scan
    f_bulk_b = np.where(f_bulk, np.round(fl * rng.uniform(2, 6, n)), 0.0)
    b_bulk_b = np.where(b_bulk, np.round(bl * rng.uniform(2, 6, n)), 0.0)
    wins = np.array([29200, 65535, 8192, 64240, 502])
    init_f = np.where(real, rng.choice(wins, n), np.where(is_scan, 1024, -1))
    init_b = np.where(real & (bp > 0), rng.choice(wins, n),
                      np.where(is_scan & (bp > 0), 0, -1))

    def period_stats(x):
        return (x, x * rng.uniform(0.0, 0.5, n), x * rng.uniform(1.0, 1.5, n),
                x * rng.uniform(0.3, 1.0, n))

    act = np.where(dur > 5, dur * rng.uniform(0.2, 1.0, n), dur)
    idle = np.where(dur > 5, dur - act, 0.0)
    a_mean, a_std, a_max, a_min = period_stats(act)
    i_mean, i_std, i_max, i_min = period_stats(idle)

    table = {
        "dst_port": port, "protocol": proto, "duration": dur,
        "fwd_pkts": fp, "bwd_pkts": bp, "fwd_bytes": fwd_bytes, "bwd_bytes": bwd_bytes,
        "flow_bytes_s": flow_bytes_s, "flow_pkts_s": tot_pkts / safe,
        "fwd_pkts_s": fp / safe, "bwd_pkts_s": bp / safe,
        "fwd_pkt_len_max": f_mx, "fwd_pkt_len_min": f_mn, "fwd_pkt_len_mean": fl, "fwd_pkt_len_std": f_sd,
        "bwd_pkt_len_max": b_mx, "bwd_pkt_len_min": b_mn, "bwd_pkt_len_mean": bl, "bwd_pkt_len_std": b_sd,
        "pkt_len_max": np.maximum(f_mx, b_mx), "pkt_len_min": np.minimum(f_mn, b_mn),
        "pkt_len_mean": p_mean, "pkt_len_std": p_sd, "pkt_len_var": p_sd**2,
        "pkt_size_avg": p_mean * rng.uniform(1.0, 1.1, n),
        "fwd_seg_size_avg": fl, "bwd_seg_size_avg": bl,
        "flow_iat_mean": fl_iat, "flow_iat_std": fl_sd, "flow_iat_max": fl_mx, "flow_iat_min": fl_mn,
        "fwd_iat_tot": np.where(fp > 1, dur, 0.0), "fwd_iat_mean": f_iat,
        "fwd_iat_std": f_isd, "fwd_iat_max": f_imx, "fwd_iat_min": f_imn,
        "bwd_iat_tot": np.where(bp > 1, dur * rng.uniform(0.6, 1.0, n), 0.0), "bwd_iat_mean": b_iat,
        "bwd_iat_std": b_isd, "bwd_iat_max": b_imx, "bwd_iat_min": b_imn,
        "fwd_psh_flags": (psh > 0).astype(int), "bwd_psh_flags": (psh > 2).astype(int),
        "fwd_urg_flags": urg, "bwd_urg_flags": np.zeros(n, dtype=int),
        "fin_flag_cnt": fin, "syn_flag_cnt": syn, "rst_flag_cnt": rst, "psh_flag_cnt": psh,
        "ack_flag_cnt": ack, "urg_flag_cnt": urg, "cwe_flag_cnt": np.zeros(n, dtype=int), "ece_flag_cnt": ece,
        "fwd_header_len": fp * hdr, "bwd_header_len": bp * hdr,
        "down_up_ratio": (bwd_bytes // np.maximum(fwd_bytes, 1)),
        "fwd_bytes_bulk_avg": f_bulk_b, "fwd_pkts_bulk_avg": np.where(f_bulk, rng.uniform(2, 8, n), 0.0),
        "fwd_bulk_rate_avg": f_bulk_b / safe,
        "bwd_bytes_bulk_avg": b_bulk_b, "bwd_pkts_bulk_avg": np.where(b_bulk, rng.uniform(2, 8, n), 0.0),
        "bwd_bulk_rate_avg": b_bulk_b / safe,
        "subflow_fwd_pkts": fp, "subflow_fwd_bytes": fwd_bytes,
        "subflow_bwd_pkts": bp, "subflow_bwd_bytes": bwd_bytes,
        "init_win_bytes_fwd": init_f, "init_win_bytes_bwd": init_b,
        "act_data_pkt_fwd": np.where(fl > 0, fp, 0), "min_seg_size_fwd": hdr,
        "active_mean": a_mean, "active_std": a_std, "active_max": a_max, "active_min": a_min,
        "idle_mean": i_mean, "idle_std": i_std, "idle_max": i_max, "idle_min": i_min,
        "label": label,
    }
    df = pd.DataFrame(table)
    assert df.shape == (n_rows, 79), df.shape
    return df.iloc[rng.permutation(n)].reset_index(drop=True)


def write_flow_csv(df, path):
    """Write the table the way a careless export would: thousands separators in bwd_bytes."""
    out = df.copy()
    out["bwd_bytes"] = out["bwd_bytes"].map("{:,}".format)
    out.to_csv(path, index=False, float_format="%.6g")

DATA_PATH = "flows.csv"
if not os.path.exists(DATA_PATH):
    write_flow_csv(make_flow_table(seed=RANDOM_STATE), DATA_PATH)
print("Data file:", DATA_PATH, f"({os.path.getsize(DATA_PATH) / 1e6:.1f} MB)")

Data file: flows.csv (50.4 MB)


In [ ]:
df = pd.read_csv(DATA_PATH)
df.iloc[:3, :8]

,dst_port,protocol,duration,fwd_pkts,bwd_pkts,fwd_bytes,bwd_bytes,flow_bytes_s
0,53,UDP,3.39516,6,5,527,"6,520",2075.600
1,80,TCP,0.15845,3,8,268,"3,728",25219.300
2,3389,TCP,120.00000,24,29,11409,"11,195",188.367


### Q1. How many rows and how many columns?

Answer with a tuple `(rows, columns)`.

In [ ]:

answer_q1 = None        # TODO: a tuple (rows, columns)
check("Q1", answer_q1)


Q1: not attempted yet (the answer is still None).


### Q2. What is the class balance, as a proportion?

Answer with proportions that sum to 1, not percentages.

In [ ]:

answer_q2 = None        # TODO: one proportion per label
check("Q2", answer_q2)


### Q3. What are the five most frequent destination ports?

Answer with the five ports, most frequent first. A Series indexed by port or a plain list both work.

In [ ]:

answer_q3 = None        # TODO: the five most frequent dst_port values, most frequent first
check("Q3", answer_q3)


### Q4. What is the mean flow duration for each class?

Answer with one mean per label.

In [ ]:

answer_q4 = None        # TODO: one mean duration per label
check("Q4", answer_q4)


### Q5. How many values are missing in each column, worst first?

Answer with a Series sorted from the most missing values to the fewest.

In [ ]:

answer_q5 = None        # TODO: missing values per column, sorted worst first
check("Q5", answer_q5)


### Q6. Name one column whose dtype is wrong, and say what it should be

Compare what each column name promises with the dtype that Pandas chose. Two text columns, `protocol` and `label`, are correct. Any other column that Pandas read as text deserves a question. `pd.api.types.is_numeric_dtype` is a useful tool here.

Answer with a tuple `(column_name, target_dtype)`, for example `("some_column", "float64")`.

In [ ]:

answer_q6 = None        # TODO: a tuple (column_name, target_dtype)
check("Q6", answer_q6)


### Fix it in code

Rule 3 says every change to the data happens in code. Convert the column you found in Q6 to the dtype you named. Look at a few of its values first: the text has a feature that Pandas cannot parse as a number. Another column in the table is meant to carry the same numbers, so you can verify your fix against it.

In [ ]:

column = None           # TODO: the column name you found in Q6, as a string
if column is not None:
    df[column] = df[column]      # TODO: replace the right-hand side with the conversion
check("Q6 fix", df[column] if column is not None else None)


### Describe the dataset in two sentences

Write them in your own words in the cell below. Say what a row is, how large the table is, how the classes are balanced, and one data-quality problem you found.


Your two sentences.

(double-click and write here)


## 8. Reproducibility rules and wrap-up

Five rules apply to every notebook you submit this semester, including the Session 11 mini-lab, and they are part of the A1 rubric.

1. Seeds: fix and record every random seed.
2. Versions: print the library versions in the first cell.
3. Code only: every change to the data happens in code, so no manual edits to the CSV.
4. Captions: every figure says what it shows.
5. Run all: the notebook works from top to bottom on a fresh runtime.

A result you cannot reproduce is not a result. In Session 8 you will deliberately break a notebook and repair it, and without seeds you could not tell whether your repair changed anything. The cell below shows what a seed does.

In [ ]:
np.random.seed(RANDOM_STATE)
first = np.random.rand(3)
np.random.seed(RANDOM_STATE)
second = np.random.rand(3)
third = np.random.rand(3)          # no reseeding: the stream simply continues

print("first :", first)
print("second:", second)
print("third :", third)
print("same seed, same numbers:", np.array_equal(first, second))
print("no reseed, new numbers :", not np.array_equal(first, third))

Write the notebook for the person who will read it without you in the room. Most of the time that person is you in three months, and a markdown cell saying why you dropped a column is worth more than the line of code that dropped it.

### Final check

Restart the runtime and run every cell from the top (Runtime > Restart session and run all). Then run the cell below. It lists the self-checks that are still open.

In [ ]:
report()

## Homework

The homework is assessed under A1.

1. Finish the six answers and the dtype fix from Section 7, and write your two-sentence description of the dataset.
2. Run the notebook end to end on a fresh runtime, and make sure that the final cell reports no open checks that you can close.
3. Download it with File > Download > Download .ipynb and submit the file.

Reading: [MLS] Chapter 6, Section 6.4 "Data pipeline architecture" (pp. 335 to 344); [Ger] Chapter 2, the data loading and quick-look sections; supplementary, the official "10 minutes to pandas" guide.

Bring one column you do not understand to Session 4. We will trace it back to the packets.

## Optional stretch

These three tasks are for students who finish early. They are not assessed.

### S1. A figure with a caption

Plot the distribution of flow duration for each class on a logarithmic scale. Use only flows with a positive duration, and write the caption in the markdown cell below the plot.

In [ ]:

positive = df[df["duration"] > 0]

fig, ax = plt.subplots(figsize=(7, 3.5))
# TODO: for each label, draw ax.hist(np.log10(duration), bins=40, alpha=0.6, density=True, label=...)
ax.set_xlabel("log10 of flow duration (seconds)")
ax.set_ylabel("density")
ax.set_title("Flow duration by class")
ax.legend()
plt.show()



Figure 2 caption.

(double-click and write here)


### S2. An engineered feature

Add a column `bytes_per_pkt` equal to the total bytes divided by the total packets in both directions, and compare its mean between the classes. Which class sends more bytes per packet?

In [ ]:

# TODO: build bytes_per_pkt without a loop, then compare its mean per label


### S3. Where do the gaps come from?

Show in code that the rows with a missing `flow_bytes_s` are exactly the rows with `duration == 0`, and write one sentence explaining why.

In [ ]:

# TODO: compare df["flow_bytes_s"].isna() with df["duration"] == 0
